# F — R-AutoEval / PPI demo (illustrative)

In [ ]:
import numpy as np, pandas as pd
from utils.csvio import load_losses_csv
from utils.testing import one_sided_binomial_pval, holm_bonferroni

csv_path='data/sample_real_losses.csv'
alpha=1.2
alpha_mtp=0.05

ids, L, cols = load_losses_csv(csv_path)
rows=[]
for i, hp in enumerate(ids):
    losses=L[i]
    n=L.shape[1]
    n_lab = n//2
    labeled = losses[:n_lab]
    unlabeled = losses[n_lab:]
    # naive 'auto-eval' proxy: mean on unlabeled
    proxy = float(unlabeled.mean())
    # bias correct with labeled mean diff (toy PPI-style idea)
    corrected = proxy + (float(labeled.mean()) - float(unlabeled.mean()))
    # calibrate with labeled data assuming [0,1] after clipping
    lab = np.clip(labeled, 0.0, 1.0)
    k = int(np.sum(lab >= alpha))  # failures if loss >= alpha (example)
    p = one_sided_binomial_pval(k, len(lab), p0=1.0-1e-6)  # placeholder tail; adapt per your loss->failure map
    rows.append({'hyperparam_id':hp,'proxy_mean':proxy,'ppi_corrected':corrected,'pval_label_calib':p})

df=pd.DataFrame(rows)
df['selected']=holm_bonferroni(df['pval_label_calib'].values, alpha=alpha_mtp)
df.sort_values(['selected','ppi_corrected'], ascending=[False,True])